# Notebook 01 — Kenya Green Finance Taxonomy (KGFT) Asset Screening Engine

## Project objective
Build an explainable first-stage screening engine for a synthetic Kenyan bank loan portfolio. This notebook preserves the original project's rule-based approach while improving the data model, screening logic, audit trail and portfolio exposure calculations.

**Important:** this is a KGFT-informed screening prototype. Keyword matches are not proof of taxonomy alignment. A final aligned decision requires activity-level evidence against the official KGFT eligibility criteria, technical screening criteria, Do No Significant Harm (DNSH) requirements and minimum social safeguards.

## Notebook sequence
1. Portfolio data and quality checks.
2. Explainable taxonomy screening rules.
3. Exclusion-first screening and evidence capture.
4. Portfolio exposure and segment analysis.
5. Export dashboard-ready screening data.

The existing notebook used a five-theme weighted keyword engine, an exclusion-first gate, a purpose-field multiplier and a TF-IDF-derived multiplier. This revision retains the five original themes and their base weights, keeps the exclusion-first logic and purpose weighting, and makes the TF-IDF adjustment exploratory rather than part of the default screening score. The next notebook will establish an independently labelled training dataset and evaluate supervised text-classification baselines. Do not train a model on the rule-generated labels and then present its performance as independent validation.


## 1. Environment setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
SEED = 42


## 2. Load the synthetic loan portfolio

In [ ]:
# In Colab, upload the CSV or mount Google Drive and update DATA_PATH.
DATA_PATH = Path("data/synthetic_kenya_loan_portfolio.csv")

if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded)))

loans = pd.read_csv(DATA_PATH, parse_dates=["origination_date"])
print("Rows:", len(loans), "| Columns:", len(loans.columns))
display(loans.head())


## 3. Validate schema and data quality

In [ ]:
EXPECTED_COLUMNS = [
    "loan_id", "origination_date", "borrower_segment", "industry_sector",
    "county", "product_description", "purpose", "collateral_type",
    "currency", "disbursed_amount", "tenor_period"
]
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(loans.columns))
assert not missing_columns, f"Missing columns: {missing_columns}"
assert loans["loan_id"].is_unique, "loan_id must be unique"
assert loans["disbursed_amount"].ge(0).all(), "Negative disbursement detected"

quality = pd.DataFrame({
    "missing_values": loans.isna().sum(),
    "missing_pct": (loans.isna().mean()*100).round(2),
    "unique_values": loans.nunique()
})
display(quality)


## 4. Review portfolio composition

In [ ]:
display(loans["borrower_segment"].value_counts().rename_axis("segment").to_frame("loans"))
display(loans["industry_sector"].value_counts().rename_axis("sector").to_frame("loans"))
display(loans.groupby("currency")["disbursed_amount"].agg(["count","sum","mean"]))


## 5. Implement the explainable KGFT-informed taxonomy engine

In [ ]:
# Place taxonomy_engine.py beside this notebook. In Colab, upload it if needed.
if not Path("taxonomy_engine.py").exists():
    from google.colab import files
    uploaded_engine = files.upload()
    assert "taxonomy_engine.py" in uploaded_engine, "Please upload taxonomy_engine.py"

from taxonomy_engine import GREEN_TAXONOMY, EXCLUSION_KEYWORDS, screen_loan, classify_portfolio

print("Taxonomy themes:", list(GREEN_TAXONOMY))
print("Exclusion terms configured:", len(EXCLUSION_KEYWORDS))


## 6. Run the first-stage screening

In [ ]:
screened = classify_portfolio(loans)
display(screened[[
    "loan_id", "industry_sector", "county", "product_description",
    "screening_category", "taxonomy_theme", "rule_score", "matched_evidence"
]].head(15))


## 7. Screening quality checks

In [ ]:
print(screened["screening_category"].value_counts(dropna=False))
assert len(screened) == len(loans)
assert screened["screening_category"].notna().all()
assert screened.loc[screened["taxonomy_theme"].eq("Potential exclusion flag"), "rule_score"].eq(0).all()


## 8. Portfolio exposure — preserve currency denomination

In [ ]:
# Do not sum KES and USD directly. Report exposures separately by currency.
exposure = (screened.groupby(["currency", "screening_category"], dropna=False)
            .agg(loan_count=("loan_id","count"),
                 disbursed_amount=("disbursed_amount","sum"))
            .reset_index())
display(exposure)


## 9. Segment and county monitoring

In [ ]:
segment_view = (screened.groupby(["borrower_segment","screening_category"])
                .agg(loan_count=("loan_id","count"),
                     disbursed_amount=("disbursed_amount","sum"))
                .reset_index())
county_view = (screened.groupby(["county","screening_category"])
               .agg(loan_count=("loan_id","count"),
                    disbursed_amount=("disbursed_amount","sum"))
               .reset_index())
display(segment_view.head(20))
display(county_view.head(20))


## 10. Create an auditable screening register

In [ ]:
screened["validation_status"] = "Not yet validated"
screened["kgft_alignment_status"] = "Not assessed"
screened["reviewer_notes"] = ""
screened["screening_method_version"] = "KGFT-informed-rules-v1.0"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
screened.to_csv(OUTPUT_DIR / "kgft_screening_register.csv", index=False)
exposure.to_csv(OUTPUT_DIR / "portfolio_exposure_by_currency.csv", index=False)
print("Saved screening register and currency-separated exposure summary.")


## 11. Interpretation and limitations

- **Potential Green — Validation Required** means the text strongly resembles a configured activity theme; it is not a final KGFT-aligned classification.
- **Manual Assessment** means the activity description is incomplete or the match is weaker.
- **Other / Insufficient evidence** means no configured positive activity term was found. It does not automatically prove that the borrower or loan is environmentally harmful.
- **Excluded / Potential exclusion** is a preliminary keyword flag. Confirm that the activity falls within the relevant official exclusion provisions.
- The original notebook’s TF-IDF multiplier was fitted on the current batch and used mean corpus-level term weights. That is not a supervised classifier, is sensitive to the batch composition, and should not be treated as a validated measure of environmental importance. It is therefore excluded from the default decision score; retain it only as an exploratory comparison.
- Amounts remain grouped by currency. Convert using a documented reporting-date FX rate before producing a single-currency portfolio total.
- Synthetic data are for technical demonstration only and must not be represented as real bank exposures.

## Next: Notebook 02 — Human-labelled data and supervised ML
Create a reviewer-labelled sample with a documented annotation guide, split train/test before fitting vectorizers, establish TF-IDF + Logistic Regression and Linear SVM baselines, and report precision, recall, F1, confusion matrix and class-level performance. Rule-generated labels are useful for bootstrapping workflow tests, not for independent model validation.
